In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import pandas as pd

# Import training data
train_df = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')

#Import test data
test_df = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')

train_df

In [ ]:
train_df.describe(include='object')

In [ ]:
train = train_df.drop(['id'], axis=1)
test = test_df.drop(['id'], axis=1)

train, test

In [ ]:
train_list = train.to_dict(orient='records')
test_list = test.to_dict(orient='records')

train_list[:5]

In [ ]:
train_txt = [str(row) for row in train_list]
test_txt = [str(row) for row in test_list]

train_txt[:5], test_txt[:5]

In [ ]:
train_df['input'] = train_txt
test_df['input'] = test_txt

train_df.input.head(), test_df.input.head()

In [ ]:
from datasets import Dataset, DatasetDict

train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

#Display training dataset
train_ds, test_ds

In [ ]:
model_nm = 'microsoft/deberta-v3-small'

from transformers import AutoModelForSequenceClassification,AutoTokenizer
tokz = AutoTokenizer.from_pretrained(model_nm)

In [ ]:
# Test tokenization
tokz.tokenize("G'day folks, I'm Reece from JCU!")

In [ ]:
# Tokenise inputs
def tok_func(x): return tokz(x["input"])

In [ ]:
train_tok_ds = train_ds.map(tok_func, batched=True)
test_tok_ds = test_ds.map(tok_func, batched=True)

# test_tok_ds[:1]

In [ ]:
train_tok_ds = train_tok_ds.rename_columns({'target':'labels'})
# test_tok_ds = test_tok_ds.rename_columns({'target':'labels'})

In [ ]:
dds = train_tok_ds.train_test_split(0.25, seed=42)

In [ ]:
from transformers import TrainingArguments,Trainer

bs = 32
epochs = 3
lr = 8e-5

In [ ]:
args = TrainingArguments(
    'outputs',
    learning_rate=lr,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    fp16=True,
    eval_strategy="epoch",
    per_device_train_batch_size=bs,
    per_device_eval_batch_size=bs*2,
    num_train_epochs=epochs,
    weight_decay=0.01,
    report_to='none'
)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    labels = p.label_ids
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds)
    }

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_nm, num_labels=2)

trainer = Trainer(
    model, args,
    train_dataset=dds['train'],
    eval_dataset=dds['test'],
    tokenizer=tokz,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train();

In [ ]:
preds = trainer.predict(test_tok_ds)
preds

In [ ]:
preds.predictions

In [ ]:
class_preds = np.argmax(preds.predictions, axis=-1)
class_preds

In [ ]:
sub = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')
sub

In [ ]:
sub['target'] = class_preds
sub

In [ ]:
name = os.path.basename(model_nm)
sub.to_csv(f'submission_{name}_ep{epochs}.csv', index=False)  
sub.to_csv(f'submission.csv', index=False)  